# Pipeline complet — Détection EPI sur chantier (YOLOv8)

Ce notebook enchaîne **toutes les étapes du pipeline**, de la préparation des données jusqu'à l'inférence avec règles de conformité, le tracking vidéo et l'application Streamlit.

Chaque étape renvoie vers le notebook détaillé correspondant et la section de documentation associée (`documentation/01.Preparation_donnees.md` à `05.Inference.md`) pour la justification complète des choix.

**Note sur la reproductibilité :** l'entraînement complet (100 epochs, ~plusieurs heures sur GPU) n'est pas relancé par défaut. La cellule d'entraînement (étape 4) est fournie telle qu'utilisée, mais désactivée via `RUN_TRAINING = False` ; le pipeline charge directement le checkpoint déjà entraîné (`runs/yolov8m_epi_1024-2/weights/best.pt`, epoch 79) pour les étapes d'évaluation et d'inférence. Les seeds sont fixées (`seed=42`) partout où un échantillonnage aléatoire est utilisé (split du dataset, visualisations).

## 0. Configuration et chemins

In [ ]:
from pathlib import Path
import torch

# Adapter selon l'environnement (local Windows ou serveur distant Linux)
PROJECT_ROOT = Path('.').resolve().parent
DATASET_ROOT = PROJECT_ROOT / 'SH17dataset'
RUNS_DIR     = PROJECT_ROOT / 'runs'
BEST_WEIGHTS = RUNS_DIR / 'yolov8m_epi_1024-2' / 'weights' / 'best.pt'

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Projet  : {PROJECT_ROOT}')
print(f'Dataset : {DATASET_ROOT}  (existe : {DATASET_ROOT.exists()})')
print(f'Device  : {DEVICE}')

## 1. Préparation des données

**Détail : `1_2_exploration.ipynb`, `1_3_pretraitement.ipynb`, `1_4_augmentation.ipynb` — voir `documentation/01.Preparation_donnees.md`.**

- Dataset source : [SH17](https://github.com/ahmadmughees/sh17dataset) (8099 images, 17 classes), téléchargé via Kaggle.
- Réorganisation (`reorganize_dataset.py`) : `images/{train,val,test}` + `labels/{train,val,test}` + `sh17.yaml`.
  Le split `test` correspond à 10 % du `val` original, tiré avec `random.seed(42)` → train=6479, val=1458, test=162.
- Pré-traitement : redimensionnement (résolution d'entrée ≥ 1024×1024, justifié par 52.3 % de petites bounding boxes < 1 % de l'image) et normalisation des pixels dans [0.0, 1.0] (gérée par le framework Ultralytics).
- Augmentation : flips, rotations, HSV jitter, mosaic — voir hyperparamètres d'entraînement à l'étape 3.

In [ ]:
import yaml

with open(DATASET_ROOT / 'sh17.yaml') as f:
    config_dataset = yaml.safe_load(f)

for split in ['train', 'val', 'test']:
    n_imgs = len(list((DATASET_ROOT / 'images' / split).glob('*.*')))
    n_lbls = len(list((DATASET_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split:5s} : {n_imgs:5d} images  /  {n_lbls:5d} labels')

print(f"\nClasses ({len(config_dataset['names'])}) : {list(config_dataset['names'].values())}")

## 2. Choix et configuration du modèle

**Détail : `2_1_choix_modele.ipynb`, `2_2_configuration.ipynb`, `2_4_entraitement_1024.ipynb` — voir `documentation/02.Choix_modele.md` et `03.Configuration_entrainement.md`.**

- Modèle retenu : **YOLOv8m** (compromis précision / vitesse / coût d'entraînement, voir doc 02).
- Résolution d'entrée retenue : **1024×1024** (vs 640×640 initialement — voir doc 03 « Révision du choix » et doc 04 section 2.4 pour la comparaison complète des deux résolutions).

## 3. Entraînement

**Détail : `2_4_entraitement_1024.ipynb` — voir `documentation/03.Configuration_entrainement.md` et `04.Evaluation.md`.**

Configuration utilisée pour produire `yolov8m_epi_1024-2/weights/best.pt` (epoch 79/100, early stopping `patience=20`). `RUN_TRAINING = False` par défaut : la cellule ci-dessous n'est **pas exécutée** dans le cadre de ce pipeline (durée de plusieurs heures sur GPU P100). Passer à `True` pour relancer l'entraînement complet.

In [ ]:
from ultralytics import YOLO

RUN_TRAINING = False

if RUN_TRAINING:
    model = YOLO('yolov8m.pt')
    results = model.train(
        data            = str(DATASET_ROOT / 'sh17.yaml'),
        epochs          = 100,
        patience        = 20,
        imgsz           = 1024,
        batch           = 8,
        device          = DEVICE,
        freeze          = 10,
        optimizer       = 'SGD',
        lr0             = 0.01,
        lrf             = 0.01,
        momentum        = 0.937,
        weight_decay    = 0.0005,
        cos_lr          = True,
        warmup_epochs   = 3,
        warmup_momentum = 0.8,
        label_smoothing = 0.1,
        hsv_h           = 0.015,
        hsv_s           = 0.7,
        hsv_v           = 0.4,
        fliplr          = 0.5,
        flipud          = 0.0,
        degrees         = 12.0,
        translate       = 0.1,
        scale           = 0.5,
        perspective     = 0.0005,
        mosaic          = 1.0,
        erasing         = 0.4,
        project         = str(RUNS_DIR),
        name            = 'yolov8m_epi_1024',
        save            = True,
        plots           = True,
    )
else:
    print('RUN_TRAINING = False — utilisation du checkpoint déjà entraîné :')
    print(BEST_WEIGHTS)

model = YOLO(str(BEST_WEIGHTS))
print(f'\nModèle chargé : {BEST_WEIGHTS}')
print(f'Classes       : {model.names}')

## 4. Évaluation sur le split test

**Détail : `2_3_evaluation.ipynb` — voir `documentation/04.Evaluation.md` section 2.4.**

Métriques attendues (modèle 1024×1024, `imgsz=1024`) : mAP50 = 0.697, mAP50-95 = 0.412, Précision = 0.798, Rappel = 0.616, F1 = 0.695.

In [ ]:
import pandas as pd

results = model.val(
    data    = str(DATASET_ROOT / 'sh17.yaml'),
    split   = 'test',
    imgsz   = 1024,
    batch   = 8,
    device  = DEVICE,
    verbose = False,
    plots   = False,
)

map50     = float(results.box.map50)
map50_95  = float(results.box.map)
precision = float(results.box.mp)
recall    = float(results.box.mr)
f1_global = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f'mAP50      : {map50:.3f}')
print(f'mAP50-95   : {map50_95:.3f}')
print(f'Précision  : {precision:.3f}')
print(f'Rappel     : {recall:.3f}')
print(f'F1 (global): {f1_global:.3f}')

In [ ]:
names       = model.names
class_idx   = results.box.ap_class_index
precision_c = results.box.p
recall_c    = results.box.r
f1_c        = results.box.f1
ap50_c      = results.box.ap50
ap_c        = results.box.ap

rows = [
    {
        'Classe':    names[i],
        'Précision': round(float(p), 3),
        'Rappel':    round(float(r), 3),
        'F1':        round(float(f), 3),
        'mAP50':     round(float(a50), 3),
        'mAP50-95':  round(float(a), 3),
    }
    for i, p, r, f, a50, a in zip(class_idx, precision_c, recall_c, f1_c, ap50_c, ap_c)
]
rows.append({
    'Classe':    'GLOBAL (moyenne)',
    'Précision': round(precision, 3),
    'Rappel':    round(recall, 3),
    'F1':        round(f1_global, 3),
    'mAP50':     round(map50, 3),
    'mAP50-95':  round(map50_95, 3),
})

table = pd.DataFrame(rows)
table

## 5. Inférence + règles de conformité EPI

**Détail : `4_1_inference_1024.ipynb` — voir `documentation/05.Inference.md`.**

Règles de conformité (recouvrement IoU entre un EPI et la partie du corps censée être protégée) :

| EPI            | Partie du corps | Message si absent           |
|----------------|------------------|------------------------------|
| `helmet`       | `head`           | casque manquant              |
| `safety-vest`  | `person`         | gilet de securite manquant   |
| `gloves`       | `hands`          | gants manquants              |

In [ ]:
import cv2

REGLES_CONFORMITE = {
    'helmet':      {'partie_corps': 'head',   'message': 'casque manquant'},
    'safety-vest': {'partie_corps': 'person', 'message': 'gilet de securite manquant'},
    'gloves':      {'partie_corps': 'hands',  'message': 'gants manquants'},
}


def iou(boite_a, boite_b):
    xa1, ya1, xa2, ya2 = boite_a
    xb1, yb1, xb2, yb2 = boite_b
    x1, y1 = max(xa1, xb1), max(ya1, yb1)
    x2, y2 = min(xa2, xb2), min(ya2, yb2)
    inter  = max(0, x2 - x1) * max(0, y2 - y1)
    aire_a = (xa2 - xa1) * (ya2 - ya1)
    aire_b = (xb2 - xb1) * (yb2 - yb1)
    union  = aire_a + aire_b - inter
    return inter / union if union > 0 else 0.0


def verifier_conformite(detections, seuil_iou=0.1):
    motifs = []
    for classe_epi, regle in REGLES_CONFORMITE.items():
        boites_partie = [d['boite'] for d in detections if d['classe'] == regle['partie_corps']]
        boites_epi    = [d['boite'] for d in detections if d['classe'] == classe_epi]
        for boite_partie in boites_partie:
            protege = any(iou(boite_partie, boite_epi) > seuil_iou for boite_epi in boites_epi)
            if not protege:
                motifs.append(regle['message'])
    return len(motifs) > 0, list(dict.fromkeys(motifs))


image_path = PROJECT_ROOT / 'photo_chantier.jpg'
image      = cv2.imread(str(image_path))

resultat   = model.predict(image, conf=0.4, device=DEVICE, verbose=False)[0]
detections = [
    {'classe': model.names[int(c)], 'boite': b.tolist(), 'confiance': float(conf)}
    for b, c, conf in zip(
        resultat.boxes.xyxy.cpu().numpy(),
        resultat.boxes.cls.cpu().numpy(),
        resultat.boxes.conf.cpu().numpy(),
    )
]

non_conforme, motifs = verifier_conformite(detections)
print(f'Image       : {image_path.name}')
print(f'Détections  : {len(detections)}')
print(f'Conformité  : {"NON CONFORME — " + ", ".join(motifs) if non_conforme else "CONFORME"}')

## 6. Tracking vidéo (Axe D)

**Détail : `5_2_tracking.ipynb` — voir `documentation/05.Inference.md`.**

Le tracking (ByteTrack natif `model.track(..., persist=True)`) suit chaque travailleur (`person`) au fil des frames et lisse le verdict de conformité sur une fenêtre glissante (15 frames, seuil 70 %) pour éviter les fausses alertes ponctuelles. Non ré-exécuté ici (vidéo source non versionnée, voir `.gitignore`) — exécuter `5_2_tracking.ipynb` directement.

## 7. Application Streamlit (Axe E)

**Détail : `5_3_streamlit.ipynb` — voir `documentation/05.Inference.md`.**

Application de démonstration permettant d'uploader une image de chantier et d'ajuster en direct le seuil de confiance (`conf_min`) et le seuil IoU (`seuil_iou`) des règles de conformité, ainsi que les EPI à vérifier. Lancée via `5_3_streamlit.ipynb` (non incluse dans ce pipeline car elle démarre un serveur persistant).